In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb as duckdb
from IPython.display import display

In [2]:
path = ("/home/adamson/Documents/Semicolon/Data Science/data_analysis/data_and_resources/InstaCart Dataset/")
aisles = pd.read_csv(path + "aisles.csv")
departments = pd.read_csv(path + "departments.csv")
order_products = pd.read_csv(path + "order_products__prior.csv")
orders = pd.read_csv(path + "orders.csv")
products = pd.read_csv(path + "products.csv")

[display(df.head()) for df in [aisles, departments,order_products, orders, products]]

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


[None, None, None, None, None]

In [12]:
# register = {
#     "aisles":aisles,
#     "departments":departments,
#     "order_products":order_products,
#     "orders":orders,
#     "products":products,
# }
# for name, df in register.items();
# duckdb.register(name, df)

In [4]:
conn = duckdb.connect()
conn.sql("""
    select *
    from products
    limit 10
""")

┌────────────┬───────────────────────────────────────────────────────────────────┬──────────┬───────────────┐
│ product_id │                           product_name                            │ aisle_id │ department_id │
│   int64    │                              varchar                              │  int64   │     int64     │
├────────────┼───────────────────────────────────────────────────────────────────┼──────────┼───────────────┤
│          1 │ Chocolate Sandwich Cookies                                        │       61 │            19 │
│          2 │ All-Seasons Salt                                                  │      104 │            13 │
│          3 │ Robust Golden Unsweetened Oolong Tea                              │       94 │             7 │
│          4 │ Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce │       38 │             1 │
│          5 │ Green Chile Anytime Sauce                                         │        5 │            13 │
│         

In [37]:
conn.sql("""
    select distinct department
    from departments
""")

┌─────────────────┐
│   department    │
│     varchar     │
├─────────────────┤
│ alcohol         │
│ beverages       │
│ pantry          │
│ dairy eggs      │
│ snacks          │
│ frozen          │
│ bakery          │
│ produce         │
│ international   │
│ pets            │
│ dry goods pasta │
│ personal care   │
│ deli            │
│ missing         │
│ other           │
│ bulk            │
│ meat seafood    │
│ breakfast       │
│ canned goods    │
│ household       │
│ babies          │
└─────────────────┘
      21 rows    

In [28]:
conn.sql("""
    select count(order_id) from orders 
""")

┌─────────────────┐
│ count(order_id) │
│      int64      │
├─────────────────┤
│         3421083 │
└─────────────────┘

No.4: List all product names found specifically in Aisle ID 61.

In [42]:
conn.sql("""
    select  product_name from products where aisle_id = 61
""")

┌────────────────────────────────────────────────────────────┐
│                        product_name                        │
│                          varchar                           │
├────────────────────────────────────────────────────────────┤
│ Chocolate Sandwich Cookies                                 │
│ Nutter Butter Cookie Bites Go-Pak                          │
│ Danish Butter Cookies                                      │
│ Gluten Free All Natural Chocolate Chip Cookies             │
│ Mini Nilla Wafers Munch Pack                               │
│ Organic Lemon Gingersnap                                   │
│ Chips Ahoy! Chewy Cookies                                  │
│ Cookie Chips Crunchy Dark Chocolate Chocolate Chip Cookies │
│ Golden Cupcakes 8 Pack                                     │
│ Crunch Vanilla Sugar Mini Cookies                          │
│               ·                                            │
│               ·                                      

No.5 : Display all orders sorted by the hour of the day they were placed (highest to lowest)

In [48]:
conn.sql("""
    select  * 
    from orders 
    order by order_hour_of_day desc
""")

┌──────────┬─────────┬──────────┬──────────────┬───────────┬───────────────────┬────────────────────────┐
│ order_id │ user_id │ eval_set │ order_number │ order_dow │ order_hour_of_day │ days_since_prior_order │
│  int64   │  int64  │ varchar  │    int64     │   int64   │       int64       │         double         │
├──────────┼─────────┼──────────┼──────────────┼───────────┼───────────────────┼────────────────────────┤
│  1977928 │   66659 │ prior    │            6 │         6 │                23 │                   28.0 │
│    20612 │   13776 │ prior    │            9 │         5 │                23 │                    7.0 │
│  1169167 │  157245 │ prior    │           10 │         6 │                23 │                    7.0 │
│   737273 │   16663 │ prior    │            1 │         6 │                23 │                   NULL │
│  1692313 │  156982 │ prior    │            1 │         5 │                23 │                   NULL │
│   807195 │   41277 │ prior    │            4

Level 2: No. 1: Calculate the total number of products associated with each department_id.

In [68]:
conn.sql("""
    select department_id, count(product_name)
    from products
    group by department_id
""")

┌───────────────┬─────────────────────┐
│ department_id │ count(product_name) │
│     int64     │        int64        │
├───────────────┼─────────────────────┤
│            11 │                6563 │
│            16 │                3449 │
│            17 │                3085 │
│             9 │                1858 │
│            14 │                1115 │
│            15 │                2092 │
│             5 │                1054 │
│            19 │                6264 │
│            13 │                5371 │
│             7 │                4365 │
│             1 │                4007 │
│             8 │                 972 │
│            21 │                1258 │
│            12 │                 907 │
│             4 │                1684 │
│             6 │                1139 │
│            20 │                1322 │
│             3 │                1516 │
│            18 │                1081 │
│             2 │                 548 │
│            10 │                  38 │


Level 2: No. 2: Find the average number of days that pass between a user's orders

In [70]:
conn.sql("""
    select user_id, avg(days_since_prior_order)
    from orders
    group by user_id
""")

┌─────────┬─────────────────────────────┐
│ user_id │ avg(days_since_prior_order) │
│  int64  │           double            │
├─────────┼─────────────────────────────┤
│    6209 │                        18.5 │
│    6232 │                         5.0 │
│    6247 │                        30.0 │
│    6260 │          17.333333333333332 │
│    6266 │          29.666666666666668 │
│    6273 │                        13.0 │
│    6283 │          23.666666666666668 │
│    6334 │                         5.5 │
│    6357 │                         8.4 │
│    6366 │                        15.0 │
│      ·  │                          ·  │
│      ·  │                          ·  │
│      ·  │                          ·  │
│   22309 │           7.730769230769231 │
│   22334 │          10.727272727272727 │
│   22348 │           9.379310344827585 │
│   22354 │                        30.0 │
│   22384 │                        16.5 │
│   22430 │           18.11111111111111 │
│   22438 │          11.7096774193

Level 2: No. 3: Count how many orders were placed on each day of the week (order_dow)

In [72]:
conn.sql("""
    select order_dow, count(order_id)
    from orders
    group by order_dow
""")

┌───────────┬─────────────────┐
│ order_dow │ count(order_id) │
│   int64   │      int64      │
├───────────┼─────────────────┤
│         3 │          436972 │
│         4 │          426339 │
│         6 │          448761 │
│         1 │          587478 │
│         5 │          453368 │
│         0 │          600905 │
│         2 │          467260 │
└───────────┴─────────────────┘

Level 2: No: 4: Identify the single busiest hour of the day for shopping based on the total number of orders.

In [73]:
conn.sql("""
    select * from orders
""")

┌──────────┬─────────┬──────────┬──────────────┬───────────┬───────────────────┬────────────────────────┐
│ order_id │ user_id │ eval_set │ order_number │ order_dow │ order_hour_of_day │ days_since_prior_order │
│  int64   │  int64  │ varchar  │    int64     │   int64   │       int64       │         double         │
├──────────┼─────────┼──────────┼──────────────┼───────────┼───────────────────┼────────────────────────┤
│  2539329 │       1 │ prior    │            1 │         2 │                 8 │                   NULL │
│  2398795 │       1 │ prior    │            2 │         3 │                 7 │                   15.0 │
│   473747 │       1 │ prior    │            3 │         3 │                12 │                   21.0 │
│  2254736 │       1 │ prior    │            4 │         4 │                 7 │                   29.0 │
│   431534 │       1 │ prior    │            5 │         4 │                15 │                   28.0 │
│  3367565 │       1 │ prior    │            6

In [85]:
conn.sql("""
    select order_hour_of_day , count(order_id) 
    from orders 
    group by order_hour_of_day 
""")

┌───────────────────┬─────────────────┐
│ order_hour_of_day │ count(order_id) │
│       int64       │      int64      │
├───────────────────┼─────────────────┤
│                12 │          272841 │
│                23 │           40043 │
│                20 │          104292 │
│                 6 │           30529 │
│                 3 │            5474 │
│                 4 │            5527 │
│                 8 │          178201 │
│                13 │          277999 │
│                19 │          140569 │
│                 1 │           12398 │
│                 · │             ·   │
│                 · │             ·   │
│                 · │             ·   │
│                14 │          283042 │
│                16 │          272553 │
│                17 │          228795 │
│                22 │           61468 │
│                 9 │          257812 │
│                 0 │           22758 │
│                 5 │            9569 │
│                10 │          288418 │


Level 2: No. 5: Find the maximum value in the add_to_cart_order column to see the largest single-
order cart size.

In [88]:
conn.sql("""
    select * from order_products limit 4
""")

┌──────────┬────────────┬───────────────────┬───────────┐
│ order_id │ product_id │ add_to_cart_order │ reordered │
│  int64   │   int64    │       int64       │   int64   │
├──────────┼────────────┼───────────────────┼───────────┤
│        2 │      33120 │                 1 │         1 │
│        2 │      28985 │                 2 │         1 │
│        2 │       9327 │                 3 │         0 │
│        2 │      45918 │                 4 │         1 │
└──────────┴────────────┴───────────────────┴───────────┘

In [91]:
conn.sql("""
    select max(add_to_cart_order) 
    from order_products
""")

┌────────────────────────┐
│ max(add_to_cart_order) │
│         int64          │
├────────────────────────┤
│                    145 │
└────────────────────────┘

Level 3: No. 1: Create a list of product names alongside their respective department names.

In [100]:
conn.sql("""
    select product_name,department 
    from products,departments;
""")

┌───────────────────────────────────────────────────────────────────────────┬────────────┐
│                               product_name                                │ department │
│                                  varchar                                  │  varchar   │
├───────────────────────────────────────────────────────────────────────────┼────────────┤
│ Chocolate Sandwich Cookies                                                │ frozen     │
│ All-Seasons Salt                                                          │ frozen     │
│ Robust Golden Unsweetened Oolong Tea                                      │ frozen     │
│ Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce         │ frozen     │
│ Green Chile Anytime Sauce                                                 │ frozen     │
│ Dry Nose Oil                                                              │ frozen     │
│ Pure Coconut Water With Orange                                            │ frozen     │

Level 3: No.2: Retrieve a list showing order_id and the names of the products contained in each order.

In [102]:
conn.sql("""
    select order_id, product_name
    from orders,products
""")

┌──────────┬────────────────────────────┐
│ order_id │        product_name        │
│  int64   │          varchar           │
├──────────┼────────────────────────────┤
│  2539329 │ Chocolate Sandwich Cookies │
│  2398795 │ Chocolate Sandwich Cookies │
│   473747 │ Chocolate Sandwich Cookies │
│  2254736 │ Chocolate Sandwich Cookies │
│   431534 │ Chocolate Sandwich Cookies │
│  3367565 │ Chocolate Sandwich Cookies │
│   550135 │ Chocolate Sandwich Cookies │
│  3108588 │ Chocolate Sandwich Cookies │
│  2295261 │ Chocolate Sandwich Cookies │
│  2550362 │ Chocolate Sandwich Cookies │
│     ·    │             ·              │
│     ·    │             ·              │
│     ·    │             ·              │
│  1906256 │ Green Chile Anytime Sauce  │
│  1721370 │ Green Chile Anytime Sauce  │
│  1318740 │ Green Chile Anytime Sauce  │
│   490161 │ Green Chile Anytime Sauce  │
│  2638175 │ Green Chile Anytime Sauce  │
│  1660930 │ Green Chile Anytime Sauce  │
│  1923886 │ Green Chile Anytime S

level 3: No.3: List all products and the name of the aisle they belong to.

In [114]:
conn.sql("""
    select product_name, aisle from products, aisles
""")

┌───────────────────────────────────────────────────────────────────────────┬────────────────────────────┐
│                               product_name                                │           aisle            │
│                                  varchar                                  │          varchar           │
├───────────────────────────────────────────────────────────────────────────┼────────────────────────────┤
│ Chocolate Sandwich Cookies                                                │ prepared soups salads      │
│ All-Seasons Salt                                                          │ prepared soups salads      │
│ Robust Golden Unsweetened Oolong Tea                                      │ prepared soups salads      │
│ Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce         │ prepared soups salads      │
│ Green Chile Anytime Sauce                                                 │ prepared soups salads      │
│ Dry Nose Oil                       

level 3: No.4: Calculate the total number of items sold per department name.

In [116]:
conn.sql("""
    select department,order_id 
    from departments, orders
    group by department, order_id
""")

┌─────────────┬──────────┐
│ department  │ order_id │
│   varchar   │  int64   │
├─────────────┼──────────┤
│ frozen      │  1916971 │
│ frozen      │   151888 │
│ frozen      │  2731325 │
│ frozen      │  1235750 │
│ frozen      │  2934168 │
│ frozen      │  1020972 │
│ frozen      │  1551437 │
│ frozen      │   545805 │
│ frozen      │  2203260 │
│ frozen      │   315191 │
│  ·          │      ·   │
│  ·          │      ·   │
│  ·          │      ·   │
│ deli        │  1399932 │
│ deli        │  1286693 │
│ deli        │  2187250 │
│ deli        │   982609 │
│ deli        │  3061897 │
│ deli        │  1975569 │
│ deli        │  2122739 │
│ deli        │   991537 │
│ deli        │    72953 │
│ missing     │   852863 │
└─────────────┴──────────┘
  ? rows       2 columns
  (>9999 rows, 20 shown) 

level 3: No.5: Display user_id, order_id, and product_name for the first 20 records in the dataset.


In [7]:
conn.sql("""
    select user_id, order_id, product_name
    from orders , products limit 20
""")

┌─────────┬──────────┬────────────────────────────┐
│ user_id │ order_id │        product_name        │
│  int64  │  int64   │          varchar           │
├─────────┼──────────┼────────────────────────────┤
│       1 │  2539329 │ Chocolate Sandwich Cookies │
│       1 │  2398795 │ Chocolate Sandwich Cookies │
│       1 │   473747 │ Chocolate Sandwich Cookies │
│       1 │  2254736 │ Chocolate Sandwich Cookies │
│       1 │   431534 │ Chocolate Sandwich Cookies │
│       1 │  3367565 │ Chocolate Sandwich Cookies │
│       1 │   550135 │ Chocolate Sandwich Cookies │
│       1 │  3108588 │ Chocolate Sandwich Cookies │
│       1 │  2295261 │ Chocolate Sandwich Cookies │
│       1 │  2550362 │ Chocolate Sandwich Cookies │
│       1 │  1187899 │ Chocolate Sandwich Cookies │
│       2 │  2168274 │ Chocolate Sandwich Cookies │
│       2 │  1501582 │ Chocolate Sandwich Cookies │
│       2 │  1901567 │ Chocolate Sandwich Cookies │
│       2 │   738281 │ Chocolate Sandwich Cookies │
│       2 │ 

Level 4: No.1: Calculate the overall reorder rate (the average of the reordered column).

In [13]:
conn.sql("""
    select avg(reordered) from order_products
""")

┌────────────────────┐
│   avg(reordered)   │
│       double       │
├────────────────────┤
│ 0.5896974667922161 │
└────────────────────┘

Level 4: No.2: Find the top 5 most popular products based on the number of times they appear in the
order_products table.


In [49]:
conn.sql("""
    select *
    --select product_name, product_id
    from order_products limit 5
""")

┌──────────┬────────────┬───────────────────┬───────────┐
│ order_id │ product_id │ add_to_cart_order │ reordered │
│  int64   │   int64    │       int64       │   int64   │
├──────────┼────────────┼───────────────────┼───────────┤
│        2 │      33120 │                 1 │         1 │
│        2 │      28985 │                 2 │         1 │
│        2 │       9327 │                 3 │         0 │
│        2 │      45918 │                 4 │         1 │
│        2 │      30035 │                 5 │         0 │
└──────────┴────────────┴───────────────────┴───────────┘

Level 4: No. 3: Identify the top 10 users who have placed the highest number of total orders.

In [17]:
conn.sql("""
    select user_id, order_number 
    --select product_name, product_id
    from orders
    order by order_number desc
    limit 10
""")

┌─────────┬──────────────┐
│ user_id │ order_number │
│  int64  │    int64     │
├─────────┼──────────────┤
│     786 │          100 │
│   12430 │          100 │
│     210 │          100 │
│     310 │          100 │
│    1310 │          100 │
│   12587 │          100 │
│     313 │          100 │
│   12473 │          100 │
│     690 │          100 │
│    1420 │          100 │
└─────────┴──────────────┘
  10 rows      2 columns

Level 4: No. 4: Find the top 5 products that are most frequently added to the cart first (where
add_to_cart_order = 1).


<!-- select 
    product_id,
    count(*) as total_orders
    from order_products
    group by product_id
    order by total_orders desc
    limit 5; -->

In [57]:
conn.sql("""
    select *
    from order_products,
    where add_to_cart_order = 1
    order by reordered desc
    limit 5
""")

┌──────────┬────────────┬───────────────────┬───────────┐
│ order_id │ product_id │ add_to_cart_order │ reordered │
│  int64   │   int64    │       int64       │   int64   │
├──────────┼────────────┼───────────────────┼───────────┤
│        2 │      33120 │                 1 │         1 │
│        5 │      13176 │                 1 │         1 │
│        8 │      23423 │                 1 │         1 │
│        3 │      33754 │                 1 │         1 │
│       10 │      24852 │                 1 │         1 │
└──────────┴────────────┴───────────────────┴───────────┘

Level 4: No. 5: Categorize orders into 'Weekend' (Days 0 and 1) vs 'Weekday' (Days 2-6) and count the total
orders for each category.

In [46]:
conn.sql("""
    select 
    case 
    when order_dow in (0, 1) then 'weekend'
    else 'weekday'
    end as category_of_day,
    count(*) as total_orders
    from orders
    group by category_of_day;
""")

┌─────────────────┬──────────────┐
│ category_of_day │ total_orders │
│     varchar     │    int64     │
├─────────────────┼──────────────┤
│ weekday         │      2232700 │
│ weekend         │      1188383 │
└─────────────────┴──────────────┘